# CC-Tempo v2 — Kaggle runner

**How to use (every time):**
1. Set `STAGE` in the next cell (see the table; follow `TODO_PLAN.md` in order). Several stages can run in one version, joined by `+` (e.g. `'A+B+C'`); they run in that order.
2. Click **Save Version → Save & Run All (Commit)**. Kaggle runs it in the background (up to 12 h; you can close the browser).
3. When the version finishes, open it → **Output** → download `cct_outputs.zip` and send it to Claude.
4. For the next stage: **+ Add Input → Your Work → this notebook (latest version)** so finished data/runs are restored, change `STAGE`, Save & Run All again.

Settings (right panel): Accelerator **GPU T4 x2**, Internet **On**. Private GitHub repo: **Add-ons → Secrets → `GH_TOKEN`** (read-only GitHub token), switched on for this notebook. Data stages (A, F1, G1) can use Accelerator *None* to save GPU quota.

| STAGE | what | GPU? | time (approx.) |
|---|---|---|---|
| `A`  | Weinreb download + LIANA + dataset | no | 1–1.5 h |
| `B`  | synthetic ground-truth benchmark | optional | 1–2 h |
| `C`  | timing calibration (50 epochs, 2 models) | yes | 10–20 min |
| `D`  | Weinreb tier-1 training (20 runs, both GPUs) | yes | 1–3 versions (resumes automatically) |
| `E`  | Weinreb evaluation + baselines + interpretation | yes | 2–4 h |
| `W2` | Weinreb tier-2 (LIANA-method robustness, controls, held-out Day 4) | yes | 1–2 versions |
| `W3` | Weinreb tier-3 (more seeds for the headline models) | yes | 1 version |
| `F1` | cytokine data prep (+ clone evidence) | no | 30–45 min |
| `F2` | cytokine training (joint + leave-one-medium-out) + eval + dose-response | yes | 1–2 versions |
| `G1` | neural-tube download + inspect + annotate | no | 30 min |
| `G2` | neural-tube LIANA + training + analysis (only after Claude has checked `inspect.txt`) | yes | 1–2 versions |

All logic lives in `scripts/kaggle_run.py` in the repo, so fixes arrive with `git clone`; the cell below only clones and calls it.


In [ ]:
STAGE = 'A+D+E'      # <-- change before each Save & Run All; join stages with '+'
EXTRA_SET = []       # optional config overrides, e.g. ['train.epochs=1000']

In [ ]:
# clone the latest code, then restore earlier work, run the stages and pack outputs (scripts/kaggle_run.py)
import os, subprocess, sys, shutil, base64
REPO = 'https://github.com/Ahnaf-Akib-Tanim/cc-tempo_v2.git'
CODE, ROOT = '/kaggle/working/cc-tempo_v2', '/kaggle/working'
auth = []
try:  # only needed if the repo is private: Kaggle secret GH_TOKEN (Add-ons -> Secrets)
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret('GH_TOKEN')
    auth = ['-c', 'http.https://github.com/.extraheader=AUTHORIZATION: basic ' + base64.b64encode(f'x-access-token:{tok}'.encode()).decode()]
except Exception:
    pass
shutil.rmtree(CODE, ignore_errors=True)
subprocess.run(['git', *auth, 'clone', '-q', REPO, CODE], check=True)
print(subprocess.run(['git', '-C', CODE, 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{CODE}/requirements-kaggle.txt'], check=True)
subprocess.run([sys.executable, f'{CODE}/scripts/kaggle_run.py', '--stage', STAGE, '--root', ROOT,
                *(['--set', *EXTRA_SET] if EXTRA_SET else [])])
os.chdir(ROOT)
shutil.rmtree(CODE, ignore_errors=True)  # code lives on GitHub; keep the version output small
print(subprocess.run(f'ls -lh {ROOT}; du -sh {ROOT}/work', shell=True, capture_output=True, text=True).stdout)